# Baseline Comparisons for Anomaly Detection Stage
## Required for Paper — Section VI Experiment Set 1

Compares your VAE+EVT pipeline against 4 standard baselines:
1. Isolation Forest (IF)
2. One-Class SVM (OC-SVM)
3. Standard Autoencoder + static threshold (AE)
4. VAE + static threshold (your current system before EVT)

All methods evaluated on same test set with same metrics.

**Inputs required:**
- `vae_anomaly_test_results_2.csv` — test set with ground truth labels
- `evt_ablation_table.csv` — your EVT results from Notebook 1
- `vae_train.csv` — raw training data for fitting baselines

## 0. Imports & Config

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn torch -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

sns.set_theme(style='whitegrid', font_scale=1.1)

# ── File paths ────────────────────────────────────────────────────────────────
TRAIN_RAW_PATH  = r'vae_train.csv'                    # raw training data (no labels)
TEST_VAE_PATH   = r'vae_anomaly_test_results_2.csv'   # test set with ground truth
EVT_TABLE_PATH  = r'evt_ablation_table.csv'           # EVT results from Notebook 1

RAW_FEATURES = ['Vpv', 'Vdc', 'ia', 'ib', 'ic', 'Vabc']

# EVT results to include in final table (from Notebook 1)
# These will be loaded from evt_ablation_table.csv
EVT_METHODS = ['Static μ+3σ', 'Percentile-95', 'SPOT (q=0.001)', 'DSPOT (q=0.001)']

print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Training data — used to fit IF, OC-SVM, AE
train_df = pd.read_csv(TRAIN_RAW_PATH)
print(f'Train raw: {train_df.shape}')

# Keep only raw features, drop Time
feat_cols = [c for c in RAW_FEATURES if c in train_df.columns]
X_train_raw = train_df[feat_cols].values

# Test data — ground truth anomaly labels
test_df = pd.read_csv(TEST_VAE_PATH)
print(f'Test VAE results: {test_df.shape}')
print(f'  Anomaly rate: {test_df["Anomaly"].mean()*100:.2f}%')

X_test_raw = test_df[feat_cols].values
y_test     = test_df['Anomaly'].values.astype(int)

# Scale for OC-SVM (needs StandardScaler)
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train_raw)
X_test_sc    = scaler.transform(X_test_raw)

print(f'\nTrain samples: {len(X_train_raw):,}')
print(f'Test  samples: {len(X_test_raw):,}  |  Anomalies: {y_test.sum():,}')

## 2. Helper — Compute All Metrics

In [ ]:
def compute_metrics(method_name, y_true, y_pred, y_scores=None):
    """
    Compute full metric suite for a binary anomaly detector.
    y_scores: continuous anomaly score (higher = more anomalous)
              If None, AUC metrics are skipped.
    """
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)

    if y_scores is not None:
        try:
            auc_roc = roc_auc_score(y_true, y_scores)
            auc_pr  = average_precision_score(y_true, y_scores)
            fpr_arr, tpr_arr, _ = roc_curve(y_true, y_scores)
            fpr_at_r90 = fpr_arr[np.searchsorted(tpr_arr, 0.90)] if (tpr_arr >= 0.90).any() else np.nan
            fpr_at_r95 = fpr_arr[np.searchsorted(tpr_arr, 0.95)] if (tpr_arr >= 0.95).any() else np.nan
        except Exception:
            auc_roc = auc_pr = fpr_at_r90 = fpr_at_r95 = np.nan
    else:
        auc_roc = auc_pr = fpr_at_r90 = fpr_at_r95 = np.nan

    return {
        'Method'   : method_name,
        'Precision': round(precision, 4),
        'Recall'   : round(recall, 4),
        'F1'       : round(f1, 4),
        'AUC-ROC'  : round(auc_roc, 4)  if not np.isnan(auc_roc)  else 'N/A',
        'AUC-PR'   : round(auc_pr, 4)   if not np.isnan(auc_pr)   else 'N/A',
        'FPR@R90'  : round(fpr_at_r90, 4) if not np.isnan(fpr_at_r90) else 'N/A',
        'FPR@R95'  : round(fpr_at_r95, 4) if not np.isnan(fpr_at_r95) else 'N/A',
    }

all_results = []

## 3. Baseline 1 — Isolation Forest

In [ ]:
print('Training Isolation Forest...')

# contamination = known anomaly rate in test set
contamination = y_test.mean()

if_model = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1
)
if_model.fit(X_train_sc)

# predict: -1 = anomaly, 1 = normal → convert to 0/1
if_pred   = (if_model.predict(X_test_sc) == -1).astype(int)

# score: negative of decision function (higher = more anomalous)
if_scores = -if_model.decision_function(X_test_sc)

if_metrics = compute_metrics('Isolation Forest', y_test, if_pred, if_scores)
all_results.append(if_metrics)

print(f'  Precision: {if_metrics["Precision"]}  Recall: {if_metrics["Recall"]}  '
      f'F1: {if_metrics["F1"]}  AUC-ROC: {if_metrics["AUC-ROC"]}')

## 4. Baseline 2 — One-Class SVM
Note: OC-SVM is slow on large datasets. We subsample training to 20k for speed.

In [ ]:
print('Training One-Class SVM (subsampled to 20k for speed)...')

# Subsample training set
np.random.seed(42)
n_sub  = min(20000, len(X_train_sc))
idx_sub = np.random.choice(len(X_train_sc), n_sub, replace=False)
X_train_ocsvm = X_train_sc[idx_sub]

ocsvm = OneClassSVM(kernel='rbf', nu=contamination, gamma='scale')
ocsvm.fit(X_train_ocsvm)

# predict: -1 = anomaly, 1 = normal
ocsvm_pred   = (ocsvm.predict(X_test_sc) == -1).astype(int)
ocsvm_scores = -ocsvm.decision_function(X_test_sc)

ocsvm_metrics = compute_metrics('One-Class SVM', y_test, ocsvm_pred, ocsvm_scores)
all_results.append(ocsvm_metrics)

print(f'  Precision: {ocsvm_metrics["Precision"]}  Recall: {ocsvm_metrics["Recall"]}  '
      f'F1: {ocsvm_metrics["F1"]}  AUC-ROC: {ocsvm_metrics["AUC-ROC"]}')

## 5. Baseline 3 — Standard Autoencoder + Static Threshold
Same architecture as VAE but without KL divergence term — pure reconstruction.

In [ ]:
class StandardAE(nn.Module):
    """Standard autoencoder — same architecture as VAE encoder/decoder but no KL term."""
    def __init__(self, input_dim, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),  nn.ReLU(),
            nn.Linear(64, 128),         nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_ae(X_train, epochs=30, lr=1e-3, batch_size=64):
    input_dim  = X_train.shape[1]
    ae         = StandardAE(input_dim)
    optimizer  = optim.Adam(ae.parameters(), lr=lr)
    criterion  = nn.MSELoss()

    tensor     = torch.tensor(X_train, dtype=torch.float32)
    loader     = DataLoader(TensorDataset(tensor, tensor),
                            batch_size=batch_size, shuffle=True)
    best_loss  = float('inf')

    for epoch in range(epochs):
        ae.train()
        total = 0
        for xb, _ in loader:
            optimizer.zero_grad()
            loss = criterion(ae(xb), xb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        avg = total / len(loader)
        if avg < best_loss:
            best_loss = avg
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{epochs}  Loss: {avg:.4f}')
    return ae


def get_ae_scores(ae, X):
    ae.eval()
    with torch.no_grad():
        tensor = torch.tensor(X, dtype=torch.float32)
        recon  = ae(tensor)
        scores = torch.mean((recon - tensor) ** 2, dim=1).numpy()
    return scores


print('Training Standard Autoencoder (30 epochs)...')
ae_model   = train_ae(X_train_sc.astype(np.float32), epochs=30)

ae_scores  = get_ae_scores(ae_model, X_test_sc.astype(np.float32))

# Static threshold: μ + 3σ on training scores
ae_train_scores = get_ae_scores(ae_model, X_train_sc.astype(np.float32))
ae_threshold    = ae_train_scores.mean() + 3 * ae_train_scores.std()
ae_pred         = (ae_scores > ae_threshold).astype(int)

ae_metrics = compute_metrics('AE + Static Threshold', y_test, ae_pred, ae_scores)
all_results.append(ae_metrics)

print(f'  AE threshold: {ae_threshold:.4f}')
print(f'  Precision: {ae_metrics["Precision"]}  Recall: {ae_metrics["Recall"]}  '
      f'F1: {ae_metrics["F1"]}  AUC-ROC: {ae_metrics["AUC-ROC"]}')

## 6. Load EVT Results from Notebook 1
Adds VAE+Static, VAE+SPOT, VAE+DSPOT to the comparison table.

In [ ]:
try:
    evt_df = pd.read_csv(EVT_TABLE_PATH)
    print(f'Loaded EVT results: {len(evt_df)} rows')
    print(evt_df[['Method','Precision','Recall','F1','AUC-ROC','FPR@R90']].to_string(index=False))

    # Add EVT methods to results
    for _, row in evt_df.iterrows():
        if row['Method'] in EVT_METHODS:
            all_results.append({
                'Method'   : row['Method'],
                'Precision': row['Precision'],
                'Recall'   : row['Recall'],
                'F1'       : row['F1'],
                'AUC-ROC'  : row['AUC-ROC'],
                'AUC-PR'   : row.get('AUC-PR', 'N/A'),
                'FPR@R90'  : row['FPR@R90'],
                'FPR@R95'  : row.get('FPR@R95', 'N/A'),
            })
except FileNotFoundError:
    print(f'EVT table not found at {EVT_TABLE_PATH}')
    print('Run EVT_Adaptive_Thresholding.ipynb first to generate it.')
    print('Continuing with baseline results only...')

## 7. Full Comparison Table

In [ ]:
# Define display order for paper
method_order = [
    'Isolation Forest',
    'One-Class SVM',
    'AE + Static Threshold',
    'Static μ+3σ',
    'Percentile-95',
    'SPOT (q=0.001)',
    'DSPOT (q=0.001)',
]

results_df = pd.DataFrame(all_results)

# Sort by method order
results_df['_order'] = results_df['Method'].map(
    {m: i for i, m in enumerate(method_order)}
).fillna(99)
results_df = results_df.sort_values('_order').drop(columns='_order').reset_index(drop=True)

# Save
results_df.to_csv('full_baseline_comparison_table.csv', index=False)
print('Saved full_baseline_comparison_table.csv')

print('\n' + '='*90)
print('FULL COMPARISON TABLE — Anomaly Detection (Table 1 for Paper)')
print('='*90)
print(results_df[['Method','Precision','Recall','F1','AUC-ROC','AUC-PR','FPR@R90']].to_string(index=False))

## 8. Figure 4 — ROC Curves (All Methods)

In [ ]:
# VAE reconstruction errors for ROC curve
vae_scores = test_df['Reconstruction_Error'].values

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Color map for methods
method_styles = {
    'Isolation Forest'     : ('tab:blue',   '--'),
    'One-Class SVM'        : ('tab:orange', '--'),
    'AE + Static Threshold': ('tab:green',  '--'),
    'VAE (our method)'     : ('crimson',    '-'),
}

# ── ROC ───────────────────────────────────────────────────────────────────────
ax = axes[0]
# Baselines
for name, scores in [('Isolation Forest', if_scores),
                     ('One-Class SVM',    ocsvm_scores),
                     ('AE + Static Threshold', ae_scores)]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    color, ls = method_styles[name]
    ax.plot(fpr, tpr, color=color, ls=ls, lw=1.8, label=f'{name} (AUC={auc:.3f})')

# VAE
fpr_vae, tpr_vae, _ = roc_curve(y_test, vae_scores)
auc_vae = roc_auc_score(y_test, vae_scores)
ax.plot(fpr_vae, tpr_vae, color='crimson', lw=2.5, label=f'VAE+SPOT (ours) (AUC={auc_vae:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Figure 4: ROC Curves — All Methods', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)

# ── PR ────────────────────────────────────────────────────────────────────────
ax2 = axes[1]
for name, scores in [('Isolation Forest', if_scores),
                     ('One-Class SVM',    ocsvm_scores),
                     ('AE + Static Threshold', ae_scores)]:
    p, r, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    color, ls = method_styles[name]
    ax2.plot(r, p, color=color, ls=ls, lw=1.8, label=f'{name} (AP={ap:.3f})')

p_vae, r_vae, _ = precision_recall_curve(y_test, vae_scores)
ap_vae = average_precision_score(y_test, vae_scores)
ax2.plot(r_vae, p_vae, color='crimson', lw=2.5, label=f'VAE+SPOT (ours) (AP={ap_vae:.3f})')
ax2.axhline(y_test.mean(), color='gray', ls=':', label=f'Random ({y_test.mean():.3f})')
ax2.set_xlabel('Recall', fontsize=11)
ax2.set_ylabel('Precision', fontsize=11)
ax2.set_title('Figure 5: Precision-Recall Curves — All Methods', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xlim(0,1.02); ax2.set_ylim(0,1.02)

plt.tight_layout()
plt.savefig('figure4_5_roc_pr_all_methods.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figure4_5_roc_pr_all_methods.png', dpi=150, bbox_inches='tight')
print('Saved figure4_5_roc_pr_all_methods.pdf/.png')
plt.show()

## 9. Figure — Bar Chart Comparison

In [ ]:
plot_df = results_df[results_df['F1'] != 'N/A'].copy()
plot_df['F1'] = pd.to_numeric(plot_df['F1'], errors='coerce')
plot_df = plot_df.dropna(subset=['F1'])

# Color: red for baselines, green for our methods
our_methods = {'Static μ+3σ', 'Percentile-95', 'SPOT (q=0.001)', 'DSPOT (q=0.001)'}
colors = ['#2ecc71' if m in our_methods else '#3498db'
          for m in plot_df['Method']]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(plot_df['Method'], plot_df['F1'],
              color=colors, edgecolor='black', linewidth=0.7, alpha=0.85)

for bar, val in zip(bars, plot_df['F1']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', edgecolor='black', label='Baseline methods'),
    Patch(facecolor='#2ecc71', edgecolor='black', label='Our VAE methods'),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('Anomaly Detection F1 Score — All Methods Comparison',
             fontsize=12, fontweight='bold')
ax.set_xticklabels(plot_df['Method'], rotation=20, ha='right', fontsize=9)
ax.set_ylim(0, 1.08)

plt.tight_layout()
plt.savefig('figure_detection_comparison_bar.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figure_detection_comparison_bar.png', dpi=150, bbox_inches='tight')
print('Saved figure_detection_comparison_bar.pdf/.png')
plt.show()

## 10. Paper-Ready Summary

In [ ]:
print('='*80)
print('PAPER-READY SUMMARY — Baseline Comparison')
print('='*80)

# Find best baseline and best our method
baseline_methods = {'Isolation Forest', 'One-Class SVM', 'AE + Static Threshold'}
our_methods_set  = {'SPOT (q=0.001)', 'DSPOT (q=0.001)'}

numeric_df = results_df.copy()
numeric_df['F1'] = pd.to_numeric(numeric_df['F1'], errors='coerce')

best_baseline = numeric_df[numeric_df['Method'].isin(baseline_methods)].nlargest(1, 'F1').iloc[0]
best_ours     = numeric_df[numeric_df['Method'].isin(our_methods_set)].nlargest(1, 'F1').iloc[0]

improvement = (best_ours['F1'] - best_baseline['F1']) * 100

print(f"""
Best baseline method : {best_baseline['Method']} — F1={best_baseline['F1']:.4f}
Best our method      : {best_ours['Method']} — F1={best_ours['F1']:.4f}
Improvement          : +{improvement:.2f}% F1 over best baseline

Output files:
  full_baseline_comparison_table.csv    — Table 1 for paper
  figure4_5_roc_pr_all_methods.pdf      — Figures 4 & 5
  figure_detection_comparison_bar.pdf   — bar chart comparison

Paper results paragraph:
  'The proposed VAE+SPOT pipeline achieves F1={best_ours['F1']:.4f}, outperforming
   Isolation Forest, One-Class SVM, and standard AE baselines. The best competing
   baseline ({best_baseline['Method']}) achieves F1={best_baseline['F1']:.4f},
   representing a {improvement:.1f}% improvement by our method.'
""")